# Data Analysis

In [1]:
import os
import sys
sys.path.append("../../modules")

from intern.ihf import utils as utils

from intern.ihf.analysis import utilities as utls

from intern.ihf.ban import extract as extract
from intern.ihf.ban import plot as plot

from intern.ihf.bd_forests import extract as extract_bdf
from intern.ihf.bd_forests import plot as plot_bdf

In [2]:
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt

In [13]:
import importlib
importlib.reload(utls)
importlib.reload(utils)
importlib.reload(extract)
importlib.reload(plot)

importlib.reload(extract_bdf)
importlib.reload(plot_bdf)

<module 'intern.ihf.bd_forests.plot' from 'c:\\Groupe_KC_AHA_Python\\main\\ihf\\../../modules\\intern\\ihf\\bd_forests\\plot.py'>

In [4]:
from intern.ihf.utils import CRS_M, D_MAX, BUFFERS, R_CLUSTER, KEY

In [5]:
adr = extract.read_concatenated_ban_files(extract.BAN_ADRESS_FILES)
occ = extract_bdf.read_bd_forests_file(extract_bdf.BD_FORESTS_FILE[0])
occ = occ[occ["exo_19"] == "BD FORET"]

c:\Groupe_KC_AHA_Python\main\ihf\../../modules\intern\ihf\ban\extract.py:40: DtypeWarning: Columns (16: nom_ld) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep=";") # sep=";" the ihf ban files are semicolon separated


In [6]:
occ = occ.to_crs(CRS_M)
adr = adr[adr.type_position.isin(["bâtiment", "entrée"])]
adr = adr[adr.numero < 99999] # drops lieux-dits (BAN convention)
pts = (gpd.GeoDataFrame(adr, geometry=gpd.points_from_xy(adr.lon, adr.lat), crs=4326)
         .to_crs(CRS_M).reset_index(drop=True))

xmin, ymin, xmax, ymax = pts.total_bounds

In [7]:
print(xmin, ymin, xmax, ymax)

808919.216116847 6214615.46390461 1072263.8937493423 6448547.308180319


In [8]:
forest_typed = (occ.cx[xmin-D_MAX:xmax+D_MAX, ymin-D_MAX:ymax+D_MAX]
                [["niv3_19", "lib3_19", "geometry"]])
forest_mask = forest_typed.dissolve().explode(index_parts=False).reset_index(drop=True)

In [9]:
forest_typed.head()

,niv3_19,lib3_19,geometry
2,311,Forets de feuillus,"POLYGON ((880356 6298566, 880356 6298536, 8803..."
3,311,Forets de feuillus,"POLYGON ((978838.669 6240454.148, 978835.274 6..."
5,311,Forets de feuillus,"POLYGON ((990230.513 6242780.372, 990210.443 6..."
7,312,Forets de coniferes,"POLYGON ((982215.504 6382577.895, 982217.215 6..."
11,312,Forets de coniferes,"POLYGON ((982903.214 6417445.214, 982895 64174..."


In [10]:
pts["d_forest"] = utls.nearest(pts, forest_mask, D_MAX)["d"]

# addresses inside forest: negative distance to the edge
inside = pts.d_forest == 0
edges = gpd.GeoDataFrame(geometry=forest_mask.boundary, crs=CRS_M)
pts.loc[inside, "d_forest"] = -utls.nearest(pts[inside], edges)["d"]

pts["forest_type_nearest"] = utls.nearest(pts, forest_typed, D_MAX,
                                          cols=("geometry", "lib3_19"))["lib3_19"]

In [11]:
for r in BUFFERS:
    pts[f"forest_{r}m"], pts[f"dom_type_{r}m"] = utls.forest_share(pts, forest_typed, r)

In [14]:
xy = np.c_[pts.geometry.x, pts.geometry.y]
pts["cluster_size"], pts["cluster"] = utls.cluster_size(xy, R_CLUSTER)

TYPO_BINS, LABELS = [0, 3, 30, np.inf], ["isolated", "grouped", "dense"]
pts["typology"] = pd.cut(pts.cluster_size, TYPO_BINS, labels=LABELS)
pts["band"] = pd.cut(pts.d_forest.fillna(np.inf), [-np.inf, 0, 50, 100, 200, np.inf],
                     labels=["in forest", "≤50 m", "50–100 m", "100–200 m", ">200 m"])
pts["ihf"] = pts.d_forest <= 100

In [15]:
for r in (30, 50, 100):
    s, _ = utls.cluster_size(xy, r)
    print(r, pd.cut(s, TYPO_BINS, labels=LABELS).value_counts().round(3).to_dict())
    

30 {'isolated': 123814, 'grouped': 157033, 'dense': 979277}
50 {'isolated': 61695, 'grouped': 87118, 'dense': 1111311}
100 {'isolated': 22463, 'grouped': 44134, 'dense': 1193527}


In [16]:
palette = {"in forest": "#1b5e20", "≤50 m": "#d73027", "50-100 m": "#fc8d59",
           "100-200 m": "#fee08b", ">200 m": "#d9d9d9"}
ct = pd.crosstab(pts.typology, pts.band, normalize="index").mul(100)
ax = ct.plot.barh(stacked=True, color=[palette[c] for c in ct.columns], figsize=(9, 3.5), width=.7)
ax.set(xlabel="% of addresses", ylabel="", title="Distance to forest edge by settlement type")
ax.legend(title="Distance", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
ax.spines[["top", "right"]].set_visible(False)

pd.crosstab(pts.typology, pts.band, margins=True).style.background_gradient(cmap="YlOrRd", axis=None)

KeyError: '50–100 m'

In [ ]:
from shapely import box
CELL = 500
pts["gx"], pts["gy"] = (pts.geometry.x // CELL).astype(int), (pts.geometry.y // CELL).astype(int)
grid = pts.groupby(["gx", "gy"]).agg(n=("ihf", "size"), p_ihf=("ihf", "mean")).reset_index()
grid = gpd.GeoDataFrame(grid, geometry=[box(x*CELL, y*CELL, (x+1)*CELL, (y+1)*CELL)
                        for x, y in zip(grid.gx, grid.gy)], crs=utils.CRS_M)
CARTO = f"https://basemaps.cartocdn.com/rastertiles/voyager/{{z}}/{{x}}/{{y}}.png?key={KEY}"
ATTR = "© OpenStreetMap contributors © CARTO"
grid[grid.n >= 5].explore("p_ihf", cmap="YlOrRd", vmin=0, vmax=1, tiles=CARTO, attr=ATTR,
                          style_kwds=dict(weight=0, fillOpacity=.75),
                          legend_kwds=dict(caption="Share of addresses ≤100 m from forest"))

In [ ]:
import folium
sub = pts[pts.nom_commune == "Allons"]
b = sub.total_bounds
m = forest_typed.cx[b[0]-300:b[2]+300, b[1]-300:b[3]+300].explore(
        column="lib3_19", cmap="Greens", tiles=CARTO, attr=ATTR,
        style_kwds=dict(weight=.3), legend=False, name="Forest")
sub[["numero", "nom_voie", "d_forest", "typology", "band", "geometry"]].explore(
        m=m, column="band", categorical=True, cmap=list(palette.values()),
        marker_kwds=dict(radius=3), tooltip=["numero", "nom_voie", "d_forest", "typology"],
        name="Addresses")
folium.LayerControl().add_to(m); m